## connection MYSQl

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

## Library Analisis

In [ ]:
import pandas as pd;
pd.set_option('display.max_rows', None)

## Mahasiswa

In [ ]:
prodiQuery = """
SELECT uuid,name FROM prodi
"""

prodiDf = pd.read_sql(prodiQuery, siakadEngine)

prodiDf

In [ ]:
jenisKeluarQuery = """
SELECT code,name FROM jenis_keluar where is_mahasiswa = 1
""";

jenisKeluarDf = pd.read_sql(jenisKeluarQuery, siakadEngine);

jenisKeluarDf

### Jumlah Mahasiwa Aktif Per Tahun dan Program Studi

In [ ]:
mahasiswaAktifQuery = """
SELECT
    semesters.year_of_study AS tahun_angkatan,
    prodi.name AS program_studi,
    faculties.name AS fakultas,
    COUNT(registers.registration_uuid) as mahasiswa_aktif

FROM mahasiswa_registers AS registers

JOIN semesters ON 
    registers.semester_code = semesters.code

JOIN prodi ON 
    registers.prodi_uuid = prodi.uuid

JOIN faculties ON
    prodi.faculty_id = faculties.id

WHERE semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 ) AND
    registers.id_jenis_keluar IS NULL AND
    registers.is_external = 0 AND
    registers.is_archive = 0 AND
    registers.deleted_at IS NULL
GROUP BY
    semesters.year_of_study,
    prodi.name,
    faculties.name
ORDER BY semesters.year_of_study DESC
"""

In [ ]:
mahasiwaAktifDf = pd.read_sql(mahasiswaAktifQuery, siakadEngine);

In [ ]:
mahasiwaAktifDf

### Mahasiswa Aktif Prodi Matematika

In [ ]:
mahasiswaAktifMatematikaDf = mahasiwaAktifDf.loc[
    mahasiwaAktifDf['program_studi'] == 'Matematika',
    ['tahun_angkatan', 'mahasiswa_aktif']
]

mahasiswaAktifMatematikaDf.style.hide(axis='index')

In [ ]:
mahasiswaAktifMatematikaPlot = mahasiswaAktifMatematikaDf.plot(
    x='tahun_angkatan',
    y='mahasiswa_aktif',
    kind='bar', 
    color='skyblue',
    edgecolor='black',
    legend=False
)

mahasiswaAktifMatematikaPlot.set_title(
    'Jumlah Mahasiswa Aktif per Tahun Angkatan',
    fontsize=14,
    pad=15
)
mahasiswaAktifMatematikaPlot.set_xlabel(
    'Tahun Angkatan',
    fontsize=12
)
mahasiswaAktifMatematikaPlot.set_ylabel(
    'Jumlah Mahasiswa',
    fontsize=12
)

mahasiswaAktifMatematikaPlot.tick_params(axis='x', rotation=0)
mahasiswaAktifMatematikaPlot.grid(axis='y', linestyle='--', alpha=0.7)

### Download Mahasiswa Aktif CSV

In [ ]:
download = mahasiwaAktifDf.to_csv('data/mahasiswa/mahasiswa-aktif.csv', index=False)

### Validasi Perhitungan Mahasiswa Aktif

In [ ]:
validasiMahasiswaAktifQuery = text("""
SELECT
    registers.nim,
    details.name as nama_mahasiswa,
    prodi.name AS program_study,
    semesters.year_of_study AS tahun_angkatan

FROM mahasiswa_registers AS registers

JOIN mahasiswa_details AS details ON
    registers.mahasiswa_uuid = details.uuid

JOIN prodi ON
    registers.prodi_uuid = prodi.uuid

JOIN semesters ON 
    registers.semester_code = semesters.code

WHERE
    registers.prodi_uuid = :prodi_uuid AND
    semesters.year_of_study = :year_of_study AND
    registers.id_jenis_keluar IS NULL AND
    registers.is_external = 0 AND
    registers.is_archive = 0 AND
    registers.deleted_at IS NULL
""")

parameter_query = {
    "prodi_uuid": "b6462f14-62ca-48e9-ada2-33b47cbe9246",
    "year_of_study": 2022
}

validasiMahasiswaDf = pd.read_sql(
    validasiMahasiswaAktifQuery, 
    siakadEngine, 
    params=parameter_query
)

total_data = len(validasiMahasiswaDf)
print(f"Total data mahasiswa: {total_data}")

### Lulusan Mahasiswa

In [ ]:
mahasiswaLulusQuery = text("""
SELECT
    registers.semester_code AS register_semester_code,
    keluar.semester_code AS keluar_semester_code,
    semesters.year_of_study AS tahun_lulus,
    prodi.name AS program_studi,
    jenis_daftar.name AS jenis_daftar,
    COUNT(keluar.id) AS jumlah_lulusan

FROM mahasiswa_keluar AS keluar

JOIN mahasiswa_keluar_details AS keluar_details ON
    keluar_details.keluar_id = keluar.id

JOIN mahasiswa_registers AS registers ON
    keluar_details.mahasiswa_register_uuid = registers.registration_uuid

JOIN prodi ON
    registers.prodi_uuid = prodi.uuid

JOIN jenis_pendaftaran AS jenis_daftar ON
    registers.jenisDaftar_code = jenis_daftar.code

JOIN semesters ON
    keluar.semester_code = semesters.code

WHERE
    keluar.jeniskeluar_code = 1 AND
     semesters.year_of_study >= ( YEAR ( CURDATE()) - 4 )

GROUP BY
    registers.semester_code,
    keluar.semester_code,
    prodi.name,
    jenis_daftar.name
ORDER BY
    keluar.semester_code DESC,
    registers.semester_code DESC
""")

mahasiswaLulusDf = pd.read_sql(mahasiswaLulusQuery, siakadEngine)

print(f"Jumlah Mahasiswa Lulus: {len(mahasiswaLulusDf)}")
mahasiswaLulusDf

### Download Mahasiswa Lulus

In [ ]:
ownload = mahasiswaLulusDf.to_csv('data/mahasiswa/mahasiswa-lulus.csv', index=False)

### Validasi Mahasiswa Lulus

In [ ]:
validasiMahasiswaLulusQuery = text("""
SELECT 
    registers.nim,
    mahasiswa.name AS mahasiswa,
    semester_keluar.code,
    semester_daftar.year_of_study AS tahun_daftar,
    semester_keluar.year_of_study AS tahun_keluar,
    semester_keluar.name,
    keluar.sk_tgl,
    prodi.name AS program_studi,
    jenis_daftar.name AS jenis_daftar,
    jenis_keluar.name AS jenis_keluar,
    registers.is_external,
    registers.is_archive,
    registers.deleted_at

FROM mahasiswa_keluar_details AS keluar_details

JOIN mahasiswa_registers AS registers ON
    keluar_details.mahasiswa_register_uuid = registers.registration_uuid

JOIN mahasiswa_keluar AS keluar ON
    keluar_details.keluar_id = keluar.id

JOIN mahasiswa_details as mahasiswa ON
    registers.mahasiswa_uuid = mahasiswa.uuid

JOIN semesters AS semester_daftar ON
    registers.semester_code = semester_daftar.code

JOIN semesters AS semester_keluar ON
    keluar.semester_code = semester_keluar.code

JOIN prodi ON
    registers.prodi_uuid = prodi.uuid

JOIN jenis_keluar ON
    registers.id_jenis_keluar = jenis_keluar.code

JOIN jenis_pendaftaran AS jenis_daftar ON
    registers.jenisDaftar_code = jenis_daftar.code

WHERE
    keluar.jeniskeluar_code = 1 AND
    prodi.uuid = :prodiUuid AND
    semester_keluar.year_of_study = :tahunKeluar AND
    semester_daftar.year_of_study = :tahunMasuk
""")

validasiMahasiswaLulusParameter =  {
    "prodiUuid": "36e17cc8-d794-4ec1-a6b9-3e2dfdf55283",
    "tahunKeluar": 2022,
    "tahunMasuk": 2018
}

validasiMahasiswaLulusDf = pd.read_sql(
    validasiMahasiswaLulusQuery,
    siakadEngine,
    params= validasiMahasiswaLulusParameter
)

total_data = len(validasiMahasiswaLulusDf)
print(f"Total data mahasiswa Lulus: {total_data}")

validasiMahasiswaLulusDf

### Eksplorasi Data

In [ ]:

summary_list = []

sumaryDf = mahasiswaLulusDf

for col in sumaryDf.select_dtypes(include=['object', 'category', 'int64']).columns:
    vc = sumaryDf[col].value_counts(dropna=False).head(10).reset_index()
    vc.columns = ['Nilai Data', 'Jumlah']
    vc.insert(0, 'Nama Kolom', col)
    
    summary_list.append(vc)

final_summary = pd.concat(summary_list, ignore_index=True)

final_summary